# Federated Learning for Breast Cancer Prediction

<div align="center">

# Federated Learning Architecture

### FedAvg — Breast Cancer Prediction

<img src="https://commons.wikimedia.org/wiki/Special:FilePath/Federated_learning.png"
     alt="Federated Learning Architecture"
     width="700">

<p><i>Federated Learning: Collaborative Training Without Sharing Raw Data</i></p>

</div>

## Project Overview
This project implements Federated Averaging (FedAvg) to train a machine learning model across three simulated hospitals using the Breast Cancer dataset.

## Objectives
- Train independent local models at each hospital.
- Establish a centralized model as a reference baseline.
- Implement FedAvg to aggregate locally trained model parameters.
- Compare model performance using Accuracy, Recall, F1-score, and ROC-AUC.

## Methodology
Each hospital trains a model using its local dataset. The server aggregates the updated model parameters using a sample-weighted average and distributes the updated global model for the next training round.

Raw training data remains in the local hospital datasets during the simulated federated training process.

## Evaluation
The final federated model is evaluated on a shared test dataset and compared with the local and centralized baselines.

## Conclusion
This project demonstrates a basic Federated Learning workflow and investigates how collaborative model training compares with independent and centralized approaches.

**Note:** This implementation simulates multiple hospitals in one Python process. It demonstrates data-local training, but does not provide complete privacy guarantees by itself.


<div align="center">

# Federated Learning for Breast Cancer Prediction

### FedAvg — Collaborative Model Training Across Three Hospitals

<img src="https://commons.wikimedia.org/wiki/Special:FilePath/Centralized%20federated%20learning%20protocol.png"
     width="750"
     alt="Federated learning architecture">

<p><i>Federated Learning Architecture — Wikimedia Commons</i></p>

</div>


In [1]:

"""
Step 3: Federated Training using FedAvg

Baselines:
A. Local models: each hospital trains independently.
B. Centralized model: pooled training data, for reference only.
C. Federated model: clients train locally and the server aggregates
   model parameters.

Outputs:
- federated_model.pkl
- results.json
"""

import json
import os

import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    roc_auc_score,
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

# Works in both a Python script and a Jupyter Notebook.
FOLDER = r"D:\My Graduation Project\Pilot Project"
os.makedirs(FOLDER, exist_ok=True)
NUM_HOSPITALS = 3
NUM_ROUNDS = 30
LOCAL_EPOCHS = 3
LEARNING_RATE = 0.01
SEED = 42

TARGET = "Diagnosis"
CLASSES = np.array([0, 1])

# IMPORTANT:
# Set this to the label representing MALIGNANT in your CSV files.
# Change to 0 if Malignant is encoded as 0.
POSITIVE_LABEL = 1


# ============================================================
# 2. DATA LOADING AND VALIDATION
# ============================================================

def load_xy(filename, expected_features=None):
    """Load a CSV file and validate its features and target."""

    path = os.path.join(FOLDER, filename)

    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Missing file: {path}\n"
            "Check that the CSV file exists in the project folder."
        )

    df = pd.read_csv(path)

    if TARGET not in df.columns:
        raise ValueError(
            f"Column '{TARGET}' not found in {filename}. "
            f"Available columns: {df.columns.tolist()}"
        )

    if df.empty:
        raise ValueError(f"{filename} is empty.")

    if df.columns.duplicated().any():
        raise ValueError(f"{filename} contains duplicate column names.")

    features = [c for c in df.columns if c != TARGET]

    if expected_features is not None:
        missing = set(expected_features) - set(features)
        extra = set(features) - set(expected_features)

        if missing or extra:
            raise ValueError(
                f"Feature mismatch in {filename}.\n"
                f"Missing: {sorted(missing)}\n"
                f"Extra: {sorted(extra)}"
            )

        # Force the same feature order in every file.
        features = expected_features

    X_df = df[features].apply(pd.to_numeric, errors="raise")
    y_series = pd.to_numeric(df[TARGET], errors="raise")

    X = X_df.to_numpy(dtype=float)
    y = y_series.to_numpy()

    if not np.isfinite(X).all():
        raise ValueError(
            f"{filename} contains missing or infinite feature values."
        )

    if pd.isna(y).any():
        raise ValueError(f"{filename} contains missing target values.")

    if not np.isin(y, CLASSES).all():
        raise ValueError(
            f"{filename} must use target labels 0 and 1."
        )

    return X, y, features


def validate_data(X_test, y_test, hospitals):
    """Check data consistency before model training."""

    if len(np.unique(y_test)) != 2:
        raise ValueError(
            "The global test set must contain both classes for ROC-AUC."
        )

    if POSITIVE_LABEL not in CLASSES:
        raise ValueError("POSITIVE_LABEL must be either 0 or 1.")

    for i, (X_h, y_h) in enumerate(hospitals, start=1):

        if len(X_h) == 0:
            raise ValueError(f"Hospital {i} has no training samples.")

        if len(X_h) != len(y_h):
            raise ValueError(
                f"Hospital {i}: X and y lengths do not match."
            )

        if X_h.shape[1] != X_test.shape[1]:
            raise ValueError(
                f"Hospital {i}: feature count does not match test data."
            )

        if len(np.unique(y_h)) != 2:
            raise ValueError(
                f"Hospital {i} must contain both classes for "
                "the local baseline in this implementation."
            )

    print("Data validation passed.")


# ============================================================
# 3. MODEL HELPERS
# ============================================================

def build_model(weights, intercept):
    """Build a fitted binary classifier from aggregated parameters."""

    model = SGDClassifier(loss="log_loss")

    model.coef_ = weights.copy()
    model.intercept_ = intercept.copy()
    model.classes_ = CLASSES.copy()
    model.n_features_in_ = weights.shape[1]

    return model


def evaluate(model, X, y):
    """Evaluate the model, treating POSITIVE_LABEL as the target class."""

    predictions = model.predict(X)
    scores = model.decision_function(X)

    # Convert labels so that the chosen positive class becomes 1.
    y_binary = (y == POSITIVE_LABEL).astype(int)
    pred_binary = (predictions == POSITIVE_LABEL).astype(int)

    # decision_function is positive toward classes_[1].
    # Reverse its sign when class 0 is the chosen positive class.
    if model.classes_[1] != POSITIVE_LABEL:
        scores = -scores

    if len(np.unique(y_binary)) != 2:
        raise ValueError(
            "Evaluation data must contain both classes for ROC-AUC."
        )

    return {
        "Accuracy": float(accuracy_score(y_binary, pred_binary)),
        "Recall": float(recall_score(y_binary, pred_binary)),
        "F1": float(f1_score(y_binary, pred_binary)),
        "AUC": float(roc_auc_score(y_binary, scores)),
    }


# ============================================================
# 4. CLIENT-SIDE TRAINING
# ============================================================

def local_train(global_w, global_b, X, y, seed):
    """Train locally from the received global model parameters."""

    model = SGDClassifier(
        loss="log_loss",
        learning_rate="constant",
        eta0=LEARNING_RATE,
        alpha=1e-4,
        random_state=seed,
    )

    # Initialize the classifier with the received global parameters.
    # Setting classes_ before partial_fit prevents the first call
    # from reinitializing these parameters.
    model.coef_ = global_w.copy()
    model.intercept_ = global_b.copy()
    model.classes_ = CLASSES.copy()
    model.n_features_in_ = X.shape[1]

    for _ in range(LOCAL_EPOCHS):
        model.partial_fit(X, y, classes=CLASSES)

    return model.coef_.copy(), model.intercept_.copy()


# ============================================================
# 5. SERVER-SIDE FEDAVG
# ============================================================

def fedavg(weights_list, intercepts_list, sizes):
    """Aggregate client parameters using sample-count weighting."""

    if not sizes or any(size <= 0 for size in sizes):
        raise ValueError("All hospital sample counts must be positive.")

    ratios = np.asarray(sizes, dtype=float)
    ratios /= ratios.sum()

    new_w = np.average(weights_list, axis=0, weights=ratios)
    new_b = np.average(intercepts_list, axis=0, weights=ratios)

    return new_w, new_b


# ============================================================
# 6. MAIN EXPERIMENT
# ============================================================

def main():

    print("Loading datasets...")

    X_test, y_test, feature_names = load_xy("global_test.csv")

    hospitals = []

    for i in range(1, NUM_HOSPITALS + 1):
        X_h, y_h, _ = load_xy(
            f"hospital_{i}.csv",
            expected_features=feature_names,
        )
        hospitals.append((X_h, y_h))

    validate_data(X_test, y_test, hospitals)

    n_features = X_test.shape[1]

    results = {
        "config": {
            "hospitals": NUM_HOSPITALS,
            "rounds": NUM_ROUNDS,
            "local_epochs": LOCAL_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "seed": SEED,
            "target": TARGET,
            "positive_label": int(POSITIVE_LABEL),
            "n_features": n_features,
        }
    }

    # --------------------------------------------------------
    # A. LOCAL MODELS
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("A) Each hospital trains alone")
    print("=" * 60)

    results["local"] = {}

    for i, (X_h, y_h) in enumerate(hospitals, start=1):

        local_model = SGDClassifier(
            loss="log_loss",
            max_iter=1000,
            random_state=SEED,
        )

        local_model.fit(X_h, y_h)

        metrics = evaluate(local_model, X_test, y_test)

        results["local"][f"Hospital {i}"] = metrics

        print(
            f"Hospital {i}: "
            f"Accuracy={metrics['Accuracy']:.3f}, "
            f"Recall={metrics['Recall']:.3f}, "
            f"F1={metrics['F1']:.3f}, "
            f"AUC={metrics['AUC']:.3f}"
        )

    # --------------------------------------------------------
    # B. CENTRALIZED BASELINE
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("B) Centralized baseline (reference only)")
    print("=" * 60)

    X_all = np.vstack([X_h for X_h, _ in hospitals])
    y_all = np.concatenate([y_h for _, y_h in hospitals])

    central = SGDClassifier(
        loss="log_loss",
        max_iter=1000,
        random_state=SEED,
    )

    central.fit(X_all, y_all)

    results["centralized"] = evaluate(central, X_test, y_test)

    print("Centralized:", results["centralized"])

    # --------------------------------------------------------
    # C. FEDERATED LEARNING: FEDAVG
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("C) Federated Learning (FedAvg)")
    print("=" * 60)

    global_w = np.zeros((1, n_features))
    global_b = np.zeros(1)

    sizes = [len(y_h) for _, y_h in hospitals]
    history = []

    for r in range(1, NUM_ROUNDS + 1):

        # Each client receives global parameters and trains locally.
        updates = [
            local_train(
                global_w,
                global_b,
                X_h,
                y_h,
                seed=SEED + r + i,
            )
            for i, (X_h, y_h) in enumerate(hospitals)
        ]

        # Server aggregates updated model parameters.
        global_w, global_b = fedavg(
            [update[0] for update in updates],
            [update[1] for update in updates],
            sizes,
        )

        global_model = build_model(global_w, global_b)
        metrics = evaluate(global_model, X_test, y_test)

        history.append({
            "Round": r,
            **metrics,
        })

        print(
            f"Round [{r:02d}/{NUM_ROUNDS}] | "
            f"Accuracy={metrics['Accuracy']:.3f} | "
            f"Recall={metrics['Recall']:.3f} | "
            f"F1={metrics['F1']:.3f} | "
            f"AUC={metrics['AUC']:.3f}"
        )

    final_model = build_model(global_w, global_b)

    results["federated"] = {
        key: value
        for key, value in history[-1].items()
        if key != "Round"
    }

    results["history"] = history

    # --------------------------------------------------------
    # 7. SAVE OUTPUTS
    # --------------------------------------------------------

    model_path = os.path.join(FOLDER, "federated_model.pkl")
    results_path = os.path.join(FOLDER, "results.json")

    joblib.dump(final_model, model_path)

    with open(results_path, "w", encoding="utf-8") as file:
        json.dump(results, file, indent=2)

    print("\nExperiment completed.")
    print("Model saved to:", model_path)
    print("Results saved to:", results_path)


if __name__ == "__main__":
    main()


Loading datasets...
Data validation passed.

A) Each hospital trains alone
Hospital 1: Accuracy=0.956, Recall=0.976, F1=0.943, AUC=0.997
Hospital 2: Accuracy=0.930, Recall=0.810, F1=0.895, AUC=0.994
Hospital 3: Accuracy=0.895, Recall=0.881, F1=0.860, AUC=0.967

B) Centralized baseline (reference only)
Centralized: {'Accuracy': 0.9824561403508771, 'Recall': 0.9761904761904762, 'F1': 0.9761904761904762, 'AUC': 0.996031746031746}

C) Federated Learning (FedAvg)
Round [01/30] | Accuracy=0.965 | Recall=0.929 | F1=0.951 | AUC=0.998
Round [02/30] | Accuracy=0.982 | Recall=0.952 | F1=0.976 | AUC=0.998
Round [03/30] | Accuracy=0.982 | Recall=0.952 | F1=0.976 | AUC=0.998
Round [04/30] | Accuracy=0.982 | Recall=0.952 | F1=0.976 | AUC=0.998
Round [05/30] | Accuracy=0.982 | Recall=0.952 | F1=0.976 | AUC=0.998
Round [06/30] | Accuracy=0.982 | Recall=0.952 | F1=0.976 | AUC=0.998
Round [07/30] | Accuracy=0.982 | Recall=0.952 | F1=0.976 | AUC=0.998
Round [08/30] | Accuracy=0.982 | Recall=0.952 | F1=0.9

In [2]:
import os
import shutil

SRC = r"C:\Users\dell\AppData\Local\Programs\Microsoft VS Code"
DST = r"D:\My Graduation Project\Pilot Project"
os.makedirs(DST, exist_ok=True)

files = [
    "breast_cancer.csv", "global_test.csv",
    "hospital_1.csv", "hospital_2.csv", "hospital_3.csv",
    "scaler.pkl", "federated_model.pkl", "results.json",
]

for f in files:
    src_path = os.path.join(SRC, f)
    if os.path.exists(src_path):
        shutil.copy(src_path, os.path.join(DST, f))
        print("copied ", f)
    else:
        print("MISSING", f)

MISSING breast_cancer.csv
MISSING global_test.csv
MISSING hospital_1.csv
MISSING hospital_2.csv
MISSING hospital_3.csv
MISSING scaler.pkl
MISSING federated_model.pkl
MISSING results.json


In [3]:
print(os.listdir(DST))

['3_app.py', 'breast_cancer.csv', 'EDA_Breast Cancer_Rahma Ashraf Ali Faragallah.ipynb', 'federated_model.pkl', 'global_test.csv', 'hospital_1.csv', 'hospital_2.csv', 'hospital_3.csv', 'Prepare_data_Rahma Ashraf Ali Faragallah.ipynb', 'requirements.txt', 'results.json', 'scaler.pkl', 'Train_Fedrated_Learning_Rahma Ashraf Ali Faragallah.ipynb']
